In [27]:
import pandas as pd
import numpy as np
import re
import os
import glob
import json
import seaborn as sns
import matplotlib.pyplot as plt

## Smoothing text
This fixes inchoerent paragraph and sentence splits to obtain the best input for our inscription extraction scheme.

In [ ]:
import re
def clean_armenian_text(text: str) -> str:
    # 1) Split text into pages (keep page numbers)
    pages = re.split(r"(---\s*Page\s+\d+\s*---)", text, flags=re.IGNORECASE)

    cleaned_pages = []

    for page in pages:
        page = page.strip()
        if not page:
            continue

        # If this is a page marker, keep as its own paragraph
        if re.match(r"---\s*Page\s+\d+\s*---", page, flags=re.IGNORECASE):
            cleaned_pages.append(page)
            continue

        # Otherwise, clean the page content
        
        # Fix hyphenated line breaks
        page = re.sub(r'-\n\s*', '', page)

        # Force paragraph breaks before lines starting with special markers
        page = re.sub(r'\n(?=(Հրատ\.|Ծանոթ\.))', r'\n\n', page)

        # Split lines to detect ALL CAPS blocks
        lines = page.splitlines()
        cleaned_lines = []
        valid_pattern = re.compile(r"^[Ա-Ֆ0-9\s\.\-,:;#\[\]\(\)]+$")

        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if not line:
                cleaned_lines.append("")
                i += 1
                continue

            # Check if this line starts an ALL CAPS block
            if valid_pattern.match(line):
                block = [line]
                i += 1
                while i < len(lines) and valid_pattern.match(lines[i].strip()):
                    block.append(lines[i].strip())
                    i += 1
                cleaned_lines.append("\n".join(block))
            else:
                # Normal text: merge lines until next blank line
                normal_para = [line]
                i += 1
                while i < len(lines) and lines[i].strip() and not valid_pattern.match(lines[i].strip()):
                    normal_para.append(lines[i].strip())
                    i += 1
                # Flatten single newlines into spaces
                para_text = " ".join(normal_para)
                para_text = re.sub(r'\s{2,}', ' ', para_text)
                cleaned_lines.append(para_text)

        cleaned_pages.append("\n\n".join(cleaned_lines))

    # Reassemble all pages with double newlines
    return "\n\n".join(cleaned_pages)




def clean_file(input_path: str, output_path: str):
    # Read original text
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Clean it
    cleaned_text = clean_armenian_text(text)

    # Write new cleaned file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned file saved as: {output_path}")


# Example usage:
clean_file("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10.txt", "divan_prak10_cleaned.txt")


Cleaned file saved as: divan_prak10_cleaned.txt


## Hybrid Method

In [113]:
import re
import string

def transform_words(strings):
    """
    Takes a list of strings and returns a flat list of words in different forms:
    original (with punctuation), upper, lower, stripped.
    """
    results = []

    for s in strings:
        words = s.split()

        for w in words:
            stripped = w.strip(string.punctuation)
            results.extend([w, w.upper(), w.lower(), stripped])

    return results


# Example usage
strings = ["Ծանոթ.", "տող.","միատող.","տողից."]
output = transform_words(strings)
output



['Ծանոթ.',
 'ԾԱՆՈԹ.',
 'ծանոթ.',
 'Ծանոթ',
 'տող.',
 'ՏՈՂ.',
 'տող.',
 'տող',
 'միատող.',
 'ՄԻԱՏՈՂ.',
 'միատող.',
 'միատող',
 'տողից.',
 'ՏՈՂԻՑ.',
 'տողից.',
 'տողից']

In [121]:
import re
import json

def is_valid_title(line: str) -> bool:
    """
    Checks if a line is a valid inscription title.
    Format: number + dot, ALL CAPS TEXT (with punctuation allowed).
    """
    line = line.strip()
    pattern = r"^\d+\.\s+[Ա-ՖԵՕՒՔ\s\.\-,:;()\[\]#]+?\."
    return bool(re.match(pattern, line))

def find_line(text, last_index, term):
    start_index = text.rfind("\n", 0, last_index)
    if start_index == -1:
        start_index = 0
    else:
        start_index += 1
    return text[start_index:last_index + len(term)]

def extract_all_caps_block(text):
    lines = text.splitlines()
    all_caps_blocks, current_block = [], []
    valid_pattern = re.compile(r"^[Ա-Ֆ0-9\s\.\-,:;#\[\]\(\)]+$")

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if current_block:
                all_caps_blocks.append("\n".join(current_block))
                current_block = []
            continue
        if valid_pattern.match(stripped):
            current_block.append(stripped)
        else:
            if current_block:
                all_caps_blocks.append("\n".join(current_block))
                current_block = []
    if current_block:
        all_caps_blocks.append("\n".join(current_block))
    return all_caps_blocks

def generate_term_ranking(text, terms_of_interest=["Ծանոթ.", "տող.","միատող.","տողից."]):
    lists = []
    terms_of_interest=transform_words(terms_of_interest)
    for term in terms_of_interest:
        matches = [m.start() for m in re.finditer(term, text, re.IGNORECASE)]
        current_list = []
        if term in transform_words(["միատող.","տող.","տողից."]): 
            for index in matches:
                line = find_line(text,index,term)
                if is_valid_title(line):
                    current_list.append((term,index))
        else:
            current_list = [(term,m) for m in matches]
        lists.append(current_list)

    flattened = [item for sublist in lists for item in sublist]
    return sorted(flattened, key=lambda x: x[1])

def extract_inscriptions_hybrid(txt_file, output_jsonl):
    results = []

    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Split into pages
    pages = re.split(r"---\s*Page\s+(\d+)\s*---", text, flags=re.IGNORECASE)
    corpus = {}
    for i in range(1, len(pages), 2):
        page_num = int(pages[i])
        page_text = pages[i+1]
        if any(keyword in page_text for keyword in transform_words(["Ծանոթ.", "տող.","Հրատ.","միատող."])):
            corpus[page_num] = page_text

    items = list(corpus.items())

    # Pass 1: deterministic
    for page_num, page_text in items:
        lines = [line.strip() for line in page_text.splitlines() if line.strip()]
        j = 0
        while j < len(lines):
            line = lines[j]
            desc_match = re.match(r"^(\d+\..*?(?:տող\.|միատող\.))", line)
            if desc_match:
                entry = {
                    "page": page_num,
                    "title": desc_match.group(1).strip(),
                    "inscription": "",
                    "publication": None,
                    "note": None
                }
                k = j+1
                inscription_lines = []
                while k < len(lines):
                    nxt = lines[k]
                    if nxt.startswith("Հրատ."):
                        entry["publication"] = nxt
                        k += 1
                        break
                    elif nxt.startswith("Ծանոթ.") or re.match(r"^\d+\.", nxt):
                        break
                    elif not re.search(r"[ա-ֆ]", nxt):
                        inscription_lines.append(nxt)
                    k += 1
                entry["inscription"] = " ".join(inscription_lines).strip()

                while k < len(lines):
                    nxt = lines[k]
                    if nxt.startswith("Ծանոթ."):
                        note_lines = [nxt]
                        m = k+1
                        while m < len(lines):
                            nxt2 = lines[m]
                            if nxt2.startswith("Հրատ.") or nxt2.startswith("Ծանոթ.") or re.match(r"^\d+\.", nxt2):
                                break
                            note_lines.append(nxt2)
                            m += 1
                        entry["note"] = " ".join(note_lines).strip()
                        k = m
                        break
                    else:
                        k += 1

                results.append(entry)
                j = k
            else:
                j += 1

    # Pass 2: fallback term-driven (to catch missed / cross-page)
    for (key1, text1), (key2, text2) in zip(items, items[1:]):
        terms1 = generate_term_ranking(text1)
        for (term1, idx1), (term2, idx2) in zip(terms1, terms1[1:]):
            if term1 in transform_words(["միատող.","տող.","տողից."]) and term2 in transform_words(["Ծանոթ."]):  # case desc → note
                inscribtion_text = text1[idx1+len(term1):idx2-1]
                publication_detection= generate_term_ranking(inscribtion_text,transform_words(["Հրատ."]))
                if len(publication_detection)>0:
                    split = inscribtion_text.split(publication_detection[0][0])
                    publication = None
                    if len(split) > 1:
                        publication = publication_detection[0][0] + split[-1]
                        inscribtion_text = split[0]

                note = text1[idx2:].split('\n\n')[0]
                title = find_line(text1, idx1, term1)
                results.append({
                    'page': key1,
                    'title': title.split('.')[0]+'.'+title.split('.')[1],
                    'inscription': inscribtion_text,
                    'publication': publication,
                    'note': note
                })

            if term1 in transform_words(["միատող.","տող.","տողից."]) and term2 in transform_words(["միատող.","տող.","տողից."]):##Case 2 desc desc
                if term1 in transform_words(['միատող.']) and term2 in transform_words(['տող.']) and idx1-3==idx2:
                    continue
                elif term2 in transform_words(['միատող.']) and term1 in transform_words(['տող.']) and idx1+3==idx2:
                    continue
                else:
                    title=find_line(text1,idx1,term1)
                    inscribtion_text=text1[idx1+len(term1):idx2-1]
                    publication = None
                    publication_detection= generate_term_ranking(inscribtion_text,transform_words(["Հրատ."]))
                    if len(publication_detection)>0:
                        split = inscribtion_text.split(publication_detection[0][0])
                        if len(split) > 1:
                            publication = publication_detection[0][0] + split[-1]
                            inscribtion_text = split[0]

                    current_entry={'page': key1,
                               'title':title.split('.')[0]+'.'+title.split('.')[1],
                                'inscribtion':" ".join(s for sub in inscribtion_text for s in sub),
                                'publication':publication,
                                'note':None}
                    results.append(current_entry)
                    
        # handle cross-page desc → note
        if key1+1 == key2:
            terms2 = generate_term_ranking(text2)
            if terms1 and terms2:
                if terms1[-1][0] in transform_words(["միատող.","տող.","տողից."]) in transform_words(["Ծանոթ."]): 
                    new_text = find_line(text1,terms1[-1][1],terms1[-1][0]) + text2
                    new_terms = generate_term_ranking(new_text)
                    idx1, idx2 = new_terms[0][1], new_terms[1][1]
                    term1, term2 = new_terms[0][0], new_terms[1][0]
                    inscribtion_text = text1[idx1+len(term1):idx2-1]

                    publication_detection= generate_term_ranking(inscribtion_text,transform_words(["Հրատ."]))
                    if len(publication_detection)>0:
                        split = inscribtion_text.split(publication_detection[0][0])
                        publication = None
                        if len(split) > 1:
                            publication = publication_detection[0][0] + split[-1]
                            inscribtion_text = split[0]
                            
                    title=find_line(new_text,idx1,term1)
                    results.append({
                        'page': key1,
                        'title':title.split('.')[0]+'.'+title.split('.')[1],
                        'inscription':inscribtion_text,
                        'publication':publication,
                        'note':note
                    })

    # Write JSONL
    with open(output_jsonl, "w", encoding="utf-8") as out:
        for entry in results:
            out.write(json.dumps(entry, ensure_ascii=False) + "\n")



In [122]:
extract_inscriptions_hybrid("divan_prak10_cleaned.txt", "divan10_cleaned_hybrid.jsonl")

## Pipeline for cleaning and extraction


In [124]:
def clean_and_extract_inscriptions(original_text, target_jsonl):
    target_cleaned=original_text.split('.txt')[0]+"_clean.txt"
    clean_file(original_text,target_cleaned)
    extract_inscriptions_hybrid(target_cleaned,target_jsonl)

    df=pd.read_json(target_jsonl,lines=True)
    df['inscribtion_number']=df['title'].apply(lambda x : int(str(x).split('.')[0]))

    df['_nan_count'] = df.drop(columns=['page', 'inscribtion_number']).isna().sum(axis=1)

    idx = df.groupby(['page', 'inscribtion_number'])['_nan_count'].idxmin()

    df_cleaned = df.loc[idx].drop(columns=['_nan_count']).reset_index(drop=True)

    df_cleaned = df_cleaned.sort_values(by='inscribtion_number').reset_index(drop=True)

    df_cleaned.to_json(target_jsonl,orient='records',lines=True,force_ascii=False)
    return df_cleaned

clean_and_extract_inscriptions("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10.txt", "pipeline_text.jsonl")


Cleaned file saved as: C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10_clean.txt


,page,title,inscription,publication,note,inscribtion,inscribtion_number
0,19,2. Ա.ՆԵԱՆ եԿԵՂԵՑԻ. հյուսիսային դռան վերնամասու...,ԱՌԱՄԱԲԱՂԻ ԱՈԱՏԱԲԱԵԽ#Ր## ՊԱՅՑԱՌԱՕԱՅԼ ԱՅՔ ԱՅՓԱՌԱ...,"Հրատ. Գէորգեանց, 127։",Ծանոթ. Վիմագիրն ամփոփում էեկեղեցու շինարարությ...,NaN,2
1,21,"3. ԵՐԵՐՈՒՅԹԻ ՏԱԾԱՐ. բեմից հյուսիս, մույթի վրա,...",Ն ՀԱՎԱՏԱՑԵԼՈՑ Ի ՔՍՆՈՐՈՅՅԵՑԻ,None,Ծանոթ. Հրատարակել է միայն ակադ. ն. Սառը (ռուսե...,NaN,3
2,22,5. ԵՐԵՐՈՒՅՔԻ ՏԱԾԱՐ. հարավային պատի արևելյան մո...,Ե# ՅԱՆՈՒՆ ԱՅՀՁՈՐԻ ԵՒ ԱՆՍԱՀԻ].. ՏԱՅՈՅԹՎԿԻՍ ԻԷՒԱՆ,None,Ծանոթ. Գրիչը չի շարունակել։,NaN,5
3,22,"6. ԵՐԵՐՈՒՅՔԻ ՏԱԾԱՐ. հյուսիսային պատին, ներքուս...",ԺՆ ՌՈԲՉԽՍՅԱՂՈՒԹՍ,None,Ծանոթ. Վերջին նախադասությունն այլ գրչով է և ան...,NaN,6
4,23,"7. ԵՐԵՐՈՒՅՔԻ ՏԱՅԱՐ. հարավային պատին, քերծագրով...",ՏՐԳՐԻԳՈՐ ՀԱՅ]ՐԱՊԵՏ ԼՈՒՍԱՒՈՐԻՉ]։,"Հրատ. Աշ. Սանուչարյան, 30, X#. 291։",Ծանոթ. Սեզ առաջին անգամ է հանդիպում Լուսավորչի...,NaN,7
...,...,...,...,...,...,...,...
134,121,"188. ՏԱՊԱՆԱԹԱՐ. օրորոցածև, հյուսիսային և հարավ...",ԽԱՈՍ ԱԱՒՆՆԵՄԵ ԱԳԱՐԱԲԱՂԴ ԳԹՈԺԱՍԱԱՅԻՆԶ ՄԱՀՈԿ ԱՆՅ...,None,None,NaN,188
135,122,"190. ՏԱՊԱՆԱՔԱՐ. օրորոցածև, հարավային նիստին, 4...","ԱՅՍ Է ՏԱՊԱՆՆ ՅՕՀԱՆԷՍԻ] ԱԲԿԱՐԵԱՆՑ, ՈՐ ԷՐ ԴԸՊԻՐԵ...",None,None,NaN,190
136,51,"562. Ա.ԱՍՏՎԱՑԱՅԻՆ ԵԿԵՂԵՑԻ. արևելյան պատին, կտի...",ՄԱՅՐ ՔՍԻ /1858։ -47-,None,None,NaN,562
137,105,"1150. ՏԱՊԱՆԱԹԱՐ. ուղղանկյուն սալ, վանքի հյուսի...",ԶՍԱՐԳԻՍ ##ԱՀԱՆԱ ՅԻՇԵՍ]ԻՏՐ,"Հրատ. Ор#б###, ######## 448.","Ծանոթ. Այս տապանագիրը մեզ չհաջողվեց գտնել, ուս...",NaN,1150


## Analysis

In [123]:
df=pd.read_json('divan10_cleaned_hybrid.jsonl',lines=True)
df['Inscribtion_Number']=df['title'].apply(lambda x : int(str(x).split('.')[0]))
df['_nan_count'] = df.drop(columns=['page', 'Inscribtion_Number']).isna().sum(axis=1)

# For each group, keep the row with the minimum NaN count
idx = df.groupby(['page', 'Inscribtion_Number'])['_nan_count'].idxmin()


df_cleaned = df.loc[idx].drop(columns=['_nan_count']).reset_index(drop=True)

# Optional: sort by Inscribtion_Number
df_cleaned = df_cleaned.sort_values(by='Inscribtion_Number').reset_index(drop=True)
df_cleaned

,page,title,inscription,publication,note,inscribtion,Inscribtion_Number
0,19,2. Ա.ՆԵԱՆ եԿԵՂԵՑԻ. հյուսիսային դռան վերնամասու...,ԱՌԱՄԱԲԱՂԻ ԱՈԱՏԱԲԱԵԽ#Ր## ՊԱՅՑԱՌԱՕԱՅԼ ԱՅՔ ԱՅՓԱՌԱ...,"Հրատ. Գէորգեանց, 127։",Ծանոթ. Վիմագիրն ամփոփում էեկեղեցու շինարարությ...,NaN,2
1,21,"3. ԵՐԵՐՈՒՅԹԻ ՏԱԾԱՐ. բեմից հյուսիս, մույթի վրա,...",Ն ՀԱՎԱՏԱՑԵԼՈՑ Ի ՔՍՆՈՐՈՅՅԵՑԻ,None,Ծանոթ. Հրատարակել է միայն ակադ. ն. Սառը (ռուսե...,NaN,3
2,22,5. ԵՐԵՐՈՒՅՔԻ ՏԱԾԱՐ. հարավային պատի արևելյան մո...,Ե# ՅԱՆՈՒՆ ԱՅՀՁՈՐԻ ԵՒ ԱՆՍԱՀԻ].. ՏԱՅՈՅԹՎԿԻՍ ԻԷՒԱՆ,None,Ծանոթ. Գրիչը չի շարունակել։,NaN,5
3,22,"6. ԵՐԵՐՈՒՅՔԻ ՏԱԾԱՐ. հյուսիսային պատին, ներքուս...",ԺՆ ՌՈԲՉԽՍՅԱՂՈՒԹՍ,None,Ծանոթ. Վերջին նախադասությունն այլ գրչով է և ան...,NaN,6
4,23,"7. ԵՐԵՐՈՒՅՔԻ ՏԱՅԱՐ. հարավային պատին, քերծագրով...",ՏՐԳՐԻԳՈՐ ՀԱՅ]ՐԱՊԵՏ ԼՈՒՍԱՒՈՐԻՉ]։,"Հրատ. Աշ. Սանուչարյան, 30, X#. 291։",Ծանոթ. Սեզ առաջին անգամ է հանդիպում Լուսավորչի...,NaN,7
...,...,...,...,...,...,...,...
134,121,"188. ՏԱՊԱՆԱԹԱՐ. օրորոցածև, հյուսիսային և հարավ...",ԽԱՈՍ ԱԱՒՆՆԵՄԵ ԱԳԱՐԱԲԱՂԴ ԳԹՈԺԱՍԱԱՅԻՆԶ ՄԱՀՈԿ ԱՆՅ...,None,None,NaN,188
135,122,"190. ՏԱՊԱՆԱՔԱՐ. օրորոցածև, հարավային նիստին, 4...","ԱՅՍ Է ՏԱՊԱՆՆ ՅՕՀԱՆԷՍԻ] ԱԲԿԱՐԵԱՆՑ, ՈՐ ԷՐ ԴԸՊԻՐԵ...",None,None,NaN,190
136,51,"562. Ա.ԱՍՏՎԱՑԱՅԻՆ ԵԿԵՂԵՑԻ. արևելյան պատին, կտի...",ՄԱՅՐ ՔՍԻ /1858։ -47-,None,None,NaN,562
137,105,"1150. ՏԱՊԱՆԱԹԱՐ. ուղղանկյուն սալ, վանքի հյուսի...",ԶՍԱՐԳԻՍ ##ԱՀԱՆԱ ՅԻՇԵՍ]ԻՏՐ,"Հրատ. Ор#б###, ######## 448.","Ծանոթ. Այս տապանագիրը մեզ չհաջողվեց գտնել, ուս...",NaN,1150


In [127]:
clean_and_extract_inscriptions("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak4.txt", "divan4.jsonl")


Cleaned file saved as: C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak4_clean.txt


,page,title,inscription,publication,note,inscribtion,inscribtion_number
0,15,"2. Ս. ԱՍՏՎԱԾԱԾԻՆ եկեղեղու- հարավային պատին, ար...",ՏՆԱՒՍԻՆԱԺԵՍԴԵՄԵԹԱՆ...... ԱԺԱՍԻՎԵՐԿԱԼԱԳԱԱԳԻՆԱՐԵ...,None,None,NaN,2
1,16,"4. ԽԱՉԲԱՐ, մեծ ու գեղեցիկ, մեկը հնագույն խաչքա...",ԳԻԹԱԿԱՆՈ ԻԹԵԱՆԱԱՈ Ի ԹԽԱԿԱՆՈՒԹԵԱՆ ՀԱՌ ՆԳԱԳԿԱԿՍԽ...,"Հրատ. Սմրատյանց, 482։ Լալաչան։ X#I#I, 117 (երկ...",Ծանոթ. Այս երևք խաչքարերը նաւյն ղերդաոտանին են...,NaN,4
2,153,"009. ԽԱՉՌԱՐ, դրանիտե, մեծ և ղեղաքանդակ, ստորին...",ԻԹՎԷՍԶ.... ՍՆ Ի ԹՎԻԱՉ. ........ ՍՆ ԴԳՍԵ..........,None,None,NaN,9
3,18,"11. ԽԱՉՔԱՐ, մեծ, գեղաքանդակ։ քիվին ուռուցիկ 1 ...",,None,Ծանոթ. Մեր հուշատետրում խաչքարի աեզադրութլունը...,NaN,11
4,18,"13. ՏԱՊԱՆԱՔԱՐ, փոքր, օրորոցաձև, բակի հարավային...",ԵՒ ԿԱԶՄՈՂ ՍԷԼԻՔՈՂԹՈՆԼ,None,None,NaN,13
...,...,...,...,...,...,...,...
368,381,"1526. ՏԱՊԱՆԱԲԱՐ, փոքր, օրորոցաձե, հյուսիսային ...",ԹՌԻԵ,None,None,NaN,1526
369,382,"1530. ՏԱՊԱՆԱՔԱՐ, տափակ, երեսին կանգնած մարդու ...",ՆԳԻՍՏ ՄԽԻԹ ՄԱԻՆՈԴԻ ՂԱՉԱՒՆ ԹՌՃԾ ԱՅՍ Է ՀԱՆԳԻՍՏ Մ...,None,None,NaN,1530
370,383,1538. Ս,NaN,None,None,,1538
371,386,"1545. ԽԱՉՔԱՐ, ընկած է եկեղեցու մուտքի առտջ, քի...",ԲԼԷ) ԱԽՈՎՆ ԱԵՆԿԱ ԱՂՈՈԿ,None,"Ծանոթ. Ժամանակակից է նախորդներին, X#II դար։ Լտ...",NaN,1545


### Where does out method fail? Where do notes not exist?
- page 26: No Note
- page 27: No Note
- Page 28: Dzanot without period
- Page 29: No Note
- Page 31: No Note
- Page 37: No Note
- Page 44: No note
- Page 46: Ցանոթ.
- Page 47: No Note


In [112]:
df_cleaned[df_cleaned['note'].isna()]

,page,title,inscription,publication,note,Inscribtion_Number
8,26,"12. ԿԱՐՄԻՐ ՎԱՆՔ ԵԿեՂԵՑԻ. հարավային պատին, արտա...","ՍՍԲԱՏԱ ԲԱԳՐԱՏՈՒՆՈՅ ՇԱՀԱՆԵԱՀԻ, ԿԱՆԳՆԵՑԱՒՎԷՍՍ ԱՅ...",None,None,12
9,27,14. ԿԱՐՄԻՐ ՎԱՆՔ ԵԿԵՂԵՑԻ. հարավային պատի ստորին...,ԵՍԳՈՒԼԱՔԸԶԻՈՐԴԷ ԸՄԵԼԻԲՍԷԹԸՍ ՈՐԵԿԻԱՅՍՍԲԴԽՄԻՍԾԱՌ...,None,None,14
11,29,16. Ա.ԿԱՐԱՊԵՏ ԵԿԵՂԵՑԻ. հյուսիսային մուտքի բարա...,ՈՎԱՐԴԱԻ7 ԳԱՐԱՈՏՆԱԵԾՅՐ ԸՐՏԴԽԱՆԻՈԽՆՏԱՅ 2ՄԵՈՑՈՊՈՆ...,None,None,16
12,31,"20. Ս. ԿԱՐԱՊԵՏ ԵԿԵՂԵՑԻ. արևելյան որմին, արտաքո...",ՍԱՆԻՆ 3 ՆՊ Ի ՆՈՅՆ ԱՒՐ ՅԱՋ ԽՈՐԱՆԻՍ,None,None,20
18,37,32. ԱՈԱՔԵԼՈՑ ԵԿԵՂԵՑԻ. հարավային խաչաթևում դրվա...,ՍԺԱԹՈ .Ս ԽԱԹՈՒ։ -33-,None,None,32
22,40,39. ԽԱՉԹԱՐԻ ՊԱՏՎԱՆԴԱՆ. Զորավորաց եկեղեցուց արև...,ՆԴՅԵԹՈՒՍԱԿԱՆԻԴԵՍՀԱՅՐՍԱՐԳԻՍԿ ԽԱՒՍՄՆԻՄՈՅԱՂԽԱՐԻՊԱ...,None,None,39
26,46,45. Ա. ԳՐԻԳՈՐ ԵԿԵՂԵՑԻ. արևելյան պատի վերին մաս...,"ՇԱԱՂԷԿԻՄԱՆՈԾ ԵՄ ՇԱՏԱՂԷԿ, ՍԻԱԲԱՆԵՑՈՆՅ ԱՅՍԲՈՒՒՄ ...",None,None,45
28,47,"47. Ա. ԳՐԻԳՈՐԵԿԵՂԵՑԻ. արևմտյան պատին, արտաքուս...",-43-,None,None,47
38,55,"64. ԳԱՎԻԹ. հյուսիսային պատին, ներքուստ, պատուհ...",Ա ԱՄԱՒԱՑՕՄՃՈԻ ՈՄԵՈԴՈՒՌԵ Հ 4 1####ԷՐՀԱԳՐԻՆԻ ՀԱԼ...,None,None,64
42,57,"68. ԳԱՎԻԹ. հյուսիսային պատին, ներքուստ, նախորդ...",ԱՒԵՏԱՐԱՆ ԶՈՒԳԱՒՔ Ի ՍԲ ԵԿԵՂԷՑԻՍ ՈՂՈՐՄՈՒԹԵԱՄԲ ՊԱ...,None,None,68


## Fuzzy

In [100]:
import re
import json
from rapidfuzz import fuzz, process

# --- Configuration ---
FUZZY_THRESHOLD = 85  # similarity % to consider a match
TERMS_OF_INTEREST = ["Ծանոթ.", "տող.", "միատող.", "տողից.", "Հրատ."]

# --- Utilities ---
def is_valid_title(line: str) -> bool:
    """
    Checks if a line is a valid inscription title.
    Format: number + dot, ALL CAPS TEXT (with punctuation allowed).
    """
    line = line.strip()
    pattern = r"^\d+\.\s+[Ա-ՖԵՕՒՔ\s\.\-,:;()\[\]#]+?\."
    return bool(re.match(pattern, line))

def find_line(text, last_index, term):
    start_index = text.rfind("\n", 0, last_index)
    if start_index == -1:
        start_index = 0
    else:
        start_index += 1
    return text[start_index:last_index + len(term)]

def extract_all_caps_block(text):
    lines = text.splitlines()
    all_caps_blocks, current_block = [], []
    valid_pattern = re.compile(r"^[Ա-Ֆ0-9\s\.\-,:;#\[\]\(\)]+$")

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if current_block:
                all_caps_blocks.append("\n".join(current_block))
                current_block = []
            continue
        if valid_pattern.match(stripped):
            current_block.append(stripped)
        else:
            if current_block:
                all_caps_blocks.append("\n".join(current_block))
                current_block = []
    if current_block:
        all_caps_blocks.append("\n".join(current_block))
    return all_caps_blocks

# --- Fuzzy term ranking ---
def generate_term_ranking_fuzzy(text, terms=TERMS_OF_INTEREST, threshold=FUZZY_THRESHOLD):
    """
    Returns a sorted list of (term, start_index) using fuzzy matching.
    Titles are validated for certain terms.
    """
    matches = []
    lines = text.splitlines()
    for idx, line in enumerate(lines):
        for term in terms:
            score = fuzz.partial_ratio(term, line)
            if score >= threshold:
                start_idx = text.find(line)
                # Only treat as title for specific terms
                if term in ["միատող.","տող.","տողից."] and is_valid_title(line):
                    matches.append((term, start_idx))
                elif term not in ["միատող.","տող.","տողից."]:
                    matches.append((term, start_idx))
    return sorted(matches, key=lambda x: x[1])

# --- Main extraction function ---
def extract_inscriptions_fuzzy(txt_file, output_jsonl):
    results = []

    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Split into pages
    pages = re.split(r"---\s*Page\s+(\d+)\s*---", text, flags=re.IGNORECASE)
    corpus = {}
    for i in range(1, len(pages), 2):
        page_num = int(pages[i])
        page_text = pages[i+1]
        if any(fuzz.partial_ratio(keyword, page_text) >= FUZZY_THRESHOLD for keyword in TERMS_OF_INTEREST):
            corpus[page_num] = page_text

    items = list(corpus.items())

    # --- Pass 1: Deterministic line-by-line extraction using fuzzy term matching ---
    for page_num, page_text in items:
        lines = [line.strip() for line in page_text.splitlines() if line.strip()]
        j = 0
        while j < len(lines):
            line = lines[j]
            # Check if line matches title pattern with fuzzy term
            if any(fuzz.partial_ratio(term, line) >= FUZZY_THRESHOLD for term in ["միատող.","տող.","տողից."]) and re.match(r"^\d+\.", line):
                entry = {
                    "page": page_num,
                    "title": line,
                    "inscription": "",
                    "publication": None,
                    "note": None
                }
                k = j + 1
                inscription_lines = []
                while k < len(lines):
                    nxt = lines[k]
                    if fuzz.partial_ratio("Հրատ.", nxt) >= FUZZY_THRESHOLD:
                        entry["publication"] = nxt
                        k += 1
                        break
                    elif fuzz.partial_ratio("Ծանոթ.", nxt) >= FUZZY_THRESHOLD or re.match(r"^\d+\.", nxt):
                        break
                    elif not re.search(r"[ա-ֆ]", nxt):
                        inscription_lines.append(nxt)
                    k += 1
                entry["inscription"] = " ".join(inscription_lines).strip()

                while k < len(lines):
                    nxt = lines[k]
                    if fuzz.partial_ratio("Ծանոթ.", nxt) >= FUZZY_THRESHOLD:
                        note_lines = [nxt]
                        m = k + 1
                        while m < len(lines):
                            nxt2 = lines[m]
                            if (fuzz.partial_ratio("Հրատ.", nxt2) >= FUZZY_THRESHOLD
                                or fuzz.partial_ratio("Ծանոթ.", nxt2) >= FUZZY_THRESHOLD
                                or re.match(r"^\d+\.", nxt2)):
                                break
                            note_lines.append(nxt2)
                            m += 1
                        entry["note"] = " ".join(note_lines).strip()
                        k = m
                        break
                    else:
                        k += 1

                results.append(entry)
                j = k
            else:
                j += 1

    # --- Pass 2: Bottom-up fallback term-driven extraction (cross-page aware) ---
    for (key1, text1), (key2, text2) in zip(items, items[1:]):
        terms1 = generate_term_ranking_fuzzy(text1)
        terms2 = generate_term_ranking_fuzzy(text2)

        # Within-page term sequences
        for (term1, idx1), (term2, idx2) in zip(terms1, terms1[1:]):
            if term1 in ["միատող.","տող.","տողից."] and term2 == "Ծանոթ.":  # desc → note
                inscrib_text = text1[idx1+len(term1):idx2].strip()
                split = [s.strip() for s in re.split("Հրատ.", inscrib_text)]
                publication = None
                if len(split) > 1:
                    publication = "Հրատ. " + split[-1]
                    inscrib_text = split[0]
                note = text1[idx2:].split('\n\n')[0]
                title = find_line(text1, idx1, term1)
                results.append({
                    "page": key1,
                    "title": title.split('.')[0]+'.'+title.split('.')[1] if '.' in title else title,
                    "inscription": inscrib_text,
                    "publication": publication,
                    "note": note
                })

        # Cross-page term sequences
        if key1 + 1 == key2 and terms1 and terms2:
            if terms1[-1][0] in ["միատող.","տող.","տողից."] and terms2[0][0] == "Ծանոթ.":
                new_text = find_line(text1, terms1[-1][1], terms1[-1][0]) + text2
                new_terms = generate_term_ranking_fuzzy(new_text)
                if len(new_terms) >= 2:
                    idx1, idx2 = new_terms[0][1], new_terms[1][1]
                    term1, term2 = new_terms[0][0], new_terms[1][0]
                    inscrib_text = new_text[idx1+len(term1):idx2].strip()
                    split = [s.strip() for s in re.split("Հրատ.", inscrib_text)]
                    publication = None
                    if len(split) > 1:
                        publication = "Հրատ. " + split[-1]
                        inscrib_text = split[0]
                    note = new_text[idx2:].split('\n\n')[0]
                    title = find_line(new_text, idx1, term1)
                    results.append({
                        "page": key1,
                        "title": title.split('.')[0]+'.'+title.split('.')[1] if '.' in title else title,
                        "inscription": inscrib_text,
                        "publication": publication,
                        "note": note
                    })

    # --- Write JSONL ---
    with open(output_jsonl, "w", encoding="utf-8") as out:
        for entry in results:
            out.write(json.dumps(entry, ensure_ascii=False) + "\n")


In [101]:
extract_inscriptions_fuzzy("divan_prak10_cleaned.txt","fuzzy.jsonl")